# FishGNN: Learning Collective Fish Dynamics via Jerk Modeling

Applies the spring-mass Jerk-GNN approach to real SLEAP pose tracking data (4 stickleback, 28 dpf, 25 fps).

## Motivation

Derivative order analysis (`fish_derivative_analysis.ipynb`) showed:
- LASSO selects velocity + acceleration only — jerk Taylor term (∝ dt³ ≈ 6×10⁻⁵) is invisible for single-step position prediction at 25 fps
- **But**: social jerk analysis shows |r| > 0.1 between fish jerk and relative neighbor velocity (v_j − v_i) for all pairs

These are compatible. Jerk is not a Taylor correction — it is the **governing equation** of social forcing:

$$j_i = f_\theta\!\left(\{v_j - v_i\},\, \{x_j - x_i\}\right)$$

The network learns how acceleration changes as a function of relative neighbor state. Position rollout follows from integrating this.

## Architecture
- **Nodes**: each fish; features = flattened position history window (k=5) + current FD velocity
- **Edges**: fully connected (social topology unknown, learned implicitly via message passing)
- **Output**: 2D acceleration **and** jerk per fish (two separate heads)
- **Hard consistency** (order 3):

$$x(t+dt) = x + v\,dt + \tfrac{1}{2}a\,dt^2 + \tfrac{1}{6}j\,dt^3$$
$$v(t+dt) = v + a\,dt + \tfrac{1}{2}j\,dt^2 \qquad a(t+dt) = a + j\,dt$$

## Loss

Normalized 3-term loss — each term dimensionless O(1); $\lambda_a$ and $\lambda_j$ are interpretable relative weights:

$$\mathcal{L} = \frac{\|x_{\text{pred}} - x_{\text{true}}\|^2}{\sigma_x^2} + \lambda_a\,\frac{\|a_{\text{pred}} - a_{\text{true}}\|^2}{\sigma_a^2} + \lambda_j\,\frac{\|j_{\text{pred}} - j_{\text{true}}\|^2}{\sigma_j^2}$$

- **Position term**: trains slow dynamics; velocity contribution dominates
- **Acceleration term**: direct supervision for the hardest-to-predict quantity; gives jerk network strong gradients without relying on O(dt³) backprop through position
- **Jerk term**: trains the social interaction law directly
- $\sigma_x^2$, $\sigma_a^2$, $\sigma_j^2$ computed from training data so initialization is scale-free

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import h5py
from torch_geometric.nn import MessagePassing

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

DATA_PATH  = '/projectnb/depaqlab/bddepasq/spring-mass/data/skeleton_28dpf_v1 complete_500_002_testinference.000_rlt_250731_v0.analysis.h5'
PNG_DIR    = '/projectnb/depaqlab/bddepasq/spring-mass/png'

FPS        = 25
DT         = 1 / FPS
FOCUS_NODE = 2        # body_1
HISTORY    = 5        # position history steps fed to GNN
LAMBDA_A   = 1.0     # relative weight on acceleration term
LAMBDA_J   = 1.0     # relative weight on jerk term
TRAIN_FRAC = 0.8      # fraction of frames used for training

# Load data
with h5py.File(DATA_PATH, 'r') as f:
    tracks      = f['tracks'][:]       # (4, 2, 6, T)
    node_names  = [n.decode() for n in f['node_names'][:]]
    track_names = [n.decode() for n in f['track_names'][:]]

# Reshape to (T, fish, xy)
pos_all = tracks.transpose(3, 0, 2, 1)          # (T, fish, node, xy)
pos     = pos_all[:, :, FOCUS_NODE, :].copy()   # (T, fish, 2)
T_total, N_FISH, _ = pos.shape
t_sec = np.arange(T_total) * DT
colors = plt.cm.tab10(np.linspace(0, 0.5, N_FISH))

print(f'Frames : {T_total}  ({T_total/FPS/60:.1f} min)')
print(f'Fish   : {N_FISH}  —  {track_names}')
print(f'Node   : {node_names[FOCUS_NODE]}')

## 1. Preprocessing

Load SLEAP HDF5 data, interpolate NaN tracking gaps, estimate v/a/j via central finite differences, split into train (80%) and test (20%) frames, and build single-step samples `(x_hist, v0) → (x_next, a_true, j_true)`.

In [ ]:
def finite_diff_vaj(pos, dt):
    """Estimate velocity, acceleration, and jerk via central finite differences."""
    v = np.gradient(pos, dt, axis=0)
    a = np.gradient(v,   dt, axis=0)
    j = np.gradient(a,   dt, axis=0)
    return v, a, j

def interpolate_nans(arr):
    """Linear interpolation over NaN gaps along axis 0."""
    out = arr.copy()
    for fish in range(arr.shape[1]):
        for d in range(arr.shape[2]):
            x = arr[:, fish, d]
            t = np.arange(len(x))
            valid = np.isfinite(x)
            if valid.sum() > 1:
                out[:, fish, d] = np.interp(t, t[valid], x[valid])
    return out

# Interpolate NaN gaps before differentiating
pos_clean = interpolate_nans(pos)
vel, acc, jrk = finite_diff_vaj(pos_clean, DT)

# Train / test split on frames (not trajectories)
T_train = int(T_total * TRAIN_FRAC)
pos_train, vel_train, acc_train, jrk_train = (
    pos_clean[:T_train], vel[:T_train], acc[:T_train], jrk[:T_train])
pos_test, vel_test, acc_test, jrk_test = (
    pos_clean[T_train:], vel[T_train:], acc[T_train:], jrk[T_train:])

print(f'Train frames: {T_train}  ({T_train/FPS:.1f}s)')
print(f'Test  frames: {T_total - T_train}  ({(T_total-T_train)/FPS:.1f}s)')

def build_dataset(pos_t, vel_t, acc_t, jrk_t, history_len):
    """Build single-step samples: (x_hist, v0) -> (x_next, a_true, j_true)."""
    dataset = []
    T = pos_t.shape[0]
    for t in range(history_len, T - 1):
        x_hist  = torch.tensor(pos_t[t-history_len:t].transpose(1, 0, 2),
                               dtype=torch.float32)   # (fish, history, 2)
        v0      = torch.tensor(vel_t[t],   dtype=torch.float32)  # (fish, 2)
        x_next  = torch.tensor(pos_t[t+1], dtype=torch.float32)  # (fish, 2)
        a_true  = torch.tensor(acc_t[t],   dtype=torch.float32)  # (fish, 2)
        j_true  = torch.tensor(jrk_t[t],  dtype=torch.float32)  # (fish, 2)
        dataset.append((x_hist, v0, x_next, a_true, j_true))
    return dataset

train_data = build_dataset(pos_train, vel_train, acc_train, jrk_train, HISTORY)
test_data  = build_dataset(pos_test,  vel_test,  acc_test,  jrk_test,  HISTORY)
print(f'Train samples: {len(train_data)}, Test samples: {len(test_data)}')

In [ ]:
# Compute normalization variances from training data (position increments, accel, jerk)
pos_increments = np.diff(pos_train, axis=0)          # (T-1, fish, 2)
sigma2_x = float(np.var(pos_increments))
sigma2_a = float(np.var(acc_train))
sigma2_j = float(np.var(jrk_train))

print(f'σ²_x  = {sigma2_x:.4f}  (px²)')
print(f'σ²_a  = {sigma2_a:.4f}  (px/s²)²')
print(f'σ²_j  = {sigma2_j:.4f}  (px/s³)²')
print(f'Ratio σ²_a / σ²_x = {sigma2_a/sigma2_x:.1f}')
print(f'Ratio σ²_j / σ²_x = {sigma2_j/sigma2_x:.1f}')

## 2. GNN Architecture

Each fish is a node. Edges are fully connected (social topology unknown). Node features: flattened 2D position history + current velocity. The network outputs both **acceleration** and **jerk** per fish. Hard consistency integration (order 3):

$$x(t+dt) = x + v \cdot dt + \tfrac{1}{2}a \cdot dt^2 + \tfrac{1}{6}j \cdot dt^3$$
$$v(t+dt) = v + a \cdot dt + \tfrac{1}{2}j \cdot dt^2 \qquad a(t+dt) = a + j \cdot dt$$

Jerk is the governing social interaction: $j_i = f_\theta(\{v_j - v_i\}, \{x_j - x_i\})$.

In [ ]:
class FishMessagePassing(MessagePassing):
    def __init__(self, node_dim, hidden_dim):
        super().__init__(aggr='sum')
        self.message_net = nn.Sequential(
            nn.Linear(2 * node_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),   nn.Tanh(),
        )
        self.update_net = nn.Sequential(
            nn.Linear(node_dim + hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, node_dim),              nn.Tanh(),
        )

    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)

    def message(self, x_i, x_j):
        return self.message_net(torch.cat([x_i, x_j], dim=-1))

    def update(self, aggr_out, x):
        return self.update_net(torch.cat([x, aggr_out], dim=-1))


class FishGNN(nn.Module):
    """
    GNN predicting 2D acceleration and jerk per fish, integrated via
    hard consistency (order 3). Fully connected edges — topology learned implicitly.
    """
    def __init__(self, n_fish, history_len, hidden_dim=64, n_layers=3):
        super().__init__()
        self.n = n_fish
        node_input_dim = history_len * 2 + 2   # history*(x,y) + (vx,vy)
        self.input_proj = nn.Linear(node_input_dim, hidden_dim)
        self.layers = nn.ModuleList([
            FishMessagePassing(hidden_dim, hidden_dim)
            for _ in range(n_layers)
        ])
        self.accel_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.Tanh(),
            nn.Linear(hidden_dim // 2, 2)
        )
        self.jerk_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.Tanh(),
            nn.Linear(hidden_dim // 2, 2)
        )

    def forward(self, x_hist, v0, a0, edge_index, dt):
        """
        x_hist: (B*n, history, 2)
        v0:     (B*n, 2)   current velocity
        a0:     (B*n, 2)   current acceleration (FD estimate)
        Returns x_next, v_next, a_next, a_pred, j_pred — all (B*n, 2)
        """
        node_feat = torch.cat([x_hist.flatten(1), v0], dim=-1)
        h = torch.tanh(self.input_proj(node_feat))
        for layer in self.layers:
            h = h + layer(h, edge_index)

        a_pred = self.accel_head(h)   # (B*n, 2)
        j_pred = self.jerk_head(h)    # (B*n, 2)

        x0 = x_hist[:, -1, :]
        x_next = x0    + v0 * dt + 0.5 * a_pred * dt**2 + (1/6) * j_pred * dt**3
        v_next = v0    + a_pred * dt + 0.5 * j_pred * dt**2
        a_next = a_pred + j_pred * dt

        return x_next, v_next, a_next, a_pred, j_pred


# Fully connected edge index for N_FISH nodes
def make_edge_index(n):
    src, dst = [], []
    for i in range(n):
        for j in range(n):
            if i != j:
                src.append(i); dst.append(j)
    return torch.tensor([src, dst], dtype=torch.long)

EDGE_INDEX = make_edge_index(N_FISH).to(device)

model = FishGNN(N_FISH, HISTORY, hidden_dim=64, n_layers=3).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')
print(f'Edge index shape: {EDGE_INDEX.shape}  ({EDGE_INDEX.shape[1]} directed edges)')

## 3. Training

Adam optimizer with cosine annealing LR schedule. Prints normalized loss breakdown (pos / σ²_x, accel / σ²_a, jerk / σ²_j) every 10 epochs so convergence of each term can be tracked independently.

In [ ]:
def collate_batch(batch, device):
    B = len(batch)
    n = N_FISH
    x_hists, v0s, x_nexts, a_trues, j_trues = zip(*batch)

    x_hist  = torch.cat(x_hists,  dim=0).to(device)   # (B*n, history, 2)
    v0      = torch.cat(v0s,      dim=0).to(device)   # (B*n, 2)
    x_next  = torch.cat(x_nexts,  dim=0).to(device)   # (B*n, 2)
    a_true  = torch.cat(a_trues,  dim=0).to(device)   # (B*n, 2)
    j_true  = torch.cat(j_trues,  dim=0).to(device)   # (B*n, 2)

    # tile edge_index across batch
    edge_index = torch.cat([EDGE_INDEX + i * n for i in range(B)], dim=1)

    return x_hist, v0, x_next, a_true, j_true, edge_index


def compute_loss(x_pred, x_next, a_pred, a_true, j_pred, j_true,
                 s2_x, s2_a, s2_j, lambda_a, lambda_j):
    """Normalized multi-term loss. Each term is dimensionless O(1)."""
    mse = nn.functional.mse_loss
    pos_term   = mse(x_pred, x_next)  / s2_x
    accel_term = mse(a_pred, a_true)  / s2_a
    jerk_term  = mse(j_pred, j_true)  / s2_j
    return pos_term + lambda_a * accel_term + lambda_j * jerk_term, pos_term, accel_term, jerk_term


def train(model, train_data, test_data,
          n_epochs=100, batch_size=256, lr=1e-3,
          lambda_a=1.0, lambda_j=1.0,
          s2_x=1.0, s2_a=1.0, s2_j=1.0):

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)

    train_losses, test_losses = [], []
    pos_losses, accel_losses, jerk_losses = [], [], []

    # Current acceleration estimate: use FD value from dataset sample
    # We need a0 for forward pass; pull from dataset (a_true of previous step)
    # For simplicity, pass zeros — a0 is not currently used as input feature
    a0_dummy = torch.zeros(batch_size * N_FISH, 2)

    for epoch in range(n_epochs):
        model.train()
        ep_total = ep_pos = ep_accel = ep_jerk = 0

        idx = torch.randperm(len(train_data))
        batches = [list(idx[i:i+batch_size].numpy())
                   for i in range(0, len(train_data), batch_size)]

        for batch_idx in batches:
            batch = [train_data[i] for i in batch_idx]
            x_hist, v0, x_next, a_true, j_true, edge_index = collate_batch(batch, device)
            a0 = torch.zeros_like(v0)   # not used as input currently

            optimizer.zero_grad()
            x_pred, _, _, a_pred, j_pred = model(x_hist, v0, a0, edge_index, DT)

            loss, pos_t, accel_t, jerk_t = compute_loss(
                x_pred, x_next, a_pred, a_true, j_pred, j_true,
                s2_x, s2_a, s2_j, lambda_a, lambda_j)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            B = len(batch_idx)
            ep_total += loss.item()    * B
            ep_pos   += pos_t.item()   * B
            ep_accel += accel_t.item() * B
            ep_jerk  += jerk_t.item()  * B

        scheduler.step()
        N = len(train_data)
        train_losses.append(ep_total / N)
        pos_losses.append(ep_pos / N)
        accel_losses.append(ep_accel / N)
        jerk_losses.append(ep_jerk / N)

        model.eval()
        with torch.no_grad():
            val_loss = 0
            for i in range(0, len(test_data), batch_size):
                batch = test_data[i:i+batch_size]
                x_hist, v0, x_next, a_true, j_true, edge_index = collate_batch(batch, device)
                a0 = torch.zeros_like(v0)
                x_pred, _, _, a_pred, j_pred = model(x_hist, v0, a0, edge_index, DT)
                vl, _, _, _ = compute_loss(x_pred, x_next, a_pred, a_true, j_pred, j_true,
                                           s2_x, s2_a, s2_j, lambda_a, lambda_j)
                val_loss += vl.item() * len(batch)
            test_losses.append(val_loss / len(test_data))

        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}/{n_epochs} | '
                  f'Total: {train_losses[-1]:.3f} | '
                  f'Pos: {pos_losses[-1]:.3f} | '
                  f'Accel: {accel_losses[-1]:.3f} | '
                  f'Jerk: {jerk_losses[-1]:.3f} | '
                  f'Test: {test_losses[-1]:.3f}')

    return train_losses, test_losses, pos_losses, accel_losses, jerk_losses


train_losses, test_losses, pos_losses, accel_losses, jerk_losses = train(
    model, train_data, test_data,
    n_epochs=100, batch_size=256, lr=1e-3,
    lambda_a=LAMBDA_A, lambda_j=LAMBDA_J,
    s2_x=sigma2_x, s2_a=sigma2_a, s2_j=sigma2_j,
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
epochs = range(1, len(train_losses) + 1)

axes[0].plot(epochs, train_losses, label='train')
axes[0].plot(epochs, test_losses,  label='test')
axes[0].set(title='Total Loss (normalized)', xlabel='Epoch')
axes[0].legend(); axes[0].set_yscale('log')

axes[1].plot(epochs, pos_losses)
axes[1].set(title='Position / σ²_x', xlabel='Epoch')
axes[1].set_yscale('log')

axes[2].plot(epochs, accel_losses)
axes[2].set(title='Accel / σ²_a', xlabel='Epoch')
axes[2].set_yscale('log')

axes[3].plot(epochs, jerk_losses)
axes[3].set(title='Jerk / σ²_j', xlabel='Epoch')
axes[3].set_yscale('log')

fig.tight_layout()
fig.savefig(f'{PNG_DIR}/fish_curves.png', dpi=150)
plt.show()

## 4. Rollout

Autoregressive rollout on held-out test frames. At each step the model predicts (a, j), applies order-3 hard consistency to get (x_next, v_next, a_next), and feeds x_next back into the position history buffer. Plots: position trajectories, accel/jerk correlation vs FD ground truth, error over time, spatial 2D paths.

In [ ]:
def rollout(model, pos_init, vel_init, n_steps, dt, edge_index, device, history_len):
    """Autoregressive rollout from initial conditions."""
    model.eval()
    pos_buf = torch.tensor(pos_init[-history_len:], dtype=torch.float32).to(device)
    v = torch.tensor(vel_init, dtype=torch.float32).to(device)  # (n_fish, 2)
    a = torch.zeros_like(v)                                      # (n_fish, 2)

    pred_pos, pred_vel, pred_accel, pred_jerk = [pos_buf[-1].cpu().numpy()], [v.cpu().numpy()], [], []

    with torch.no_grad():
        for _ in range(n_steps):
            x_hist = pos_buf.unsqueeze(0).permute(1, 0, 2)  # (n_fish, history, 2)
            x_next, v_next, a_next, a_pred, j_pred = model(x_hist, v, a, edge_index, dt)

            pred_pos.append(x_next.cpu().numpy())
            pred_vel.append(v_next.cpu().numpy())
            pred_accel.append(a_pred.cpu().numpy())
            pred_jerk.append(j_pred.cpu().numpy())

            pos_buf = torch.cat([pos_buf[1:], x_next.unsqueeze(0)], dim=0)
            v, a = v_next, a_next

    return (np.array(pred_pos),    # (n_steps+1, n_fish, 2)
            np.array(pred_vel),    # (n_steps+1, n_fish, 2)
            np.array(pred_accel),  # (n_steps,   n_fish, 2)
            np.array(pred_jerk))   # (n_steps,   n_fish, 2)


TRIM    = 10
ROLLOUT = 500

start = HISTORY + TRIM
pred_pos, pred_vel, pred_accel, pred_jerk = rollout(
    model,
    pos_test[start - HISTORY : start],
    vel_test[start],
    ROLLOUT, DT, EDGE_INDEX, device, HISTORY
)
true_pos   = pos_test[start : start + ROLLOUT + 1]
true_accel = acc_test[start : start + ROLLOUT]
true_jerk  = jrk_test[start : start + ROLLOUT]
t_roll     = np.arange(ROLLOUT + 1) * DT
print(f'Rollout: {ROLLOUT} steps ({ROLLOUT*DT:.1f}s)')

In [ ]:
fig, axes = plt.subplots(N_FISH, 2, figsize=(12, 3 * N_FISH), sharex=True)
for fi in range(N_FISH):
    for di, dim in enumerate(['x', 'y']):
        ax = axes[fi, di]
        ax.plot(t_roll, true_pos[:, fi, di], color=colors[fi], label='true')
        ax.plot(t_roll, pred_pos[:, fi, di], '--', color=colors[fi], label='pred')
        ax.set(ylabel=f'fish {fi+1} {dim} (px)')
        if fi == 0:
            ax.legend(); ax.set_title(f'Position {dim}')
axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].set_xlabel('Time (s)')
fig.tight_layout()
fig.savefig(f'{PNG_DIR}/fish_rollout.png', dpi=150)
plt.show()

In [ ]:
from scipy.stats import pearsonr

fig, axes = plt.subplots(N_FISH, 4, figsize=(16, 3 * N_FISH))
for fi in range(N_FISH):
    for di, dim in enumerate(['x', 'y']):
        # Acceleration correlation
        ax = axes[fi, di]
        t = true_accel[:, fi, di]; p = pred_accel[:, fi, di]
        r, _ = pearsonr(t, p)
        ax.scatter(t, p, s=2, alpha=0.3, color=colors[fi])
        lim = max(np.abs(t).max(), np.abs(p).max())
        ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8)
        ax.set(xlabel='true accel', ylabel='pred accel',
               title=f'fish {fi+1} accel {dim}  r={r:.3f}')

        # Jerk correlation
        ax = axes[fi, di + 2]
        t = true_jerk[:, fi, di]; p = pred_jerk[:, fi, di]
        r, _ = pearsonr(t, p)
        ax.scatter(t, p, s=2, alpha=0.3, color=colors[fi])
        lim = max(np.abs(t).max(), np.abs(p).max())
        ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8)
        ax.set(xlabel='true jerk', ylabel='pred jerk',
               title=f'fish {fi+1} jerk {dim}  r={r:.3f}')

fig.tight_layout()
fig.savefig(f'{PNG_DIR}/fish_corr.png', dpi=150)
plt.show()

In [ ]:
pos_err   = np.sqrt(((pred_pos   - true_pos  )**2).sum(axis=-1))   # (T+1, fish)
accel_err = np.sqrt(((pred_accel - true_accel)**2).sum(axis=-1))   # (T,   fish)
jerk_err  = np.sqrt(((pred_jerk  - true_jerk )**2).sum(axis=-1))   # (T,   fish)

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)

for fi in range(N_FISH):
    axes[0].plot(t_roll,        pos_err[:,  fi], color=colors[fi], label=f'fish {fi+1}')
    axes[1].plot(t_roll[:-1],   accel_err[:, fi], color=colors[fi])
    axes[2].plot(t_roll[:-1],   jerk_err[:,  fi], color=colors[fi])

axes[0].set(ylabel='Position RMSE (px)',    title='Position Error')
axes[1].set(ylabel='Accel RMSE (px/s²)',    title='Acceleration Error')
axes[2].set(ylabel='Jerk RMSE (px/s³)',     title='Jerk Error', xlabel='Time (s)')
axes[0].legend()

fig.tight_layout()
fig.savefig(f'{PNG_DIR}/fish_error.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, N_FISH, figsize=(4 * N_FISH, 4))
for fi in range(N_FISH):
    ax = axes[fi]
    ax.plot(true_pos[:, fi, 0], true_pos[:, fi, 1],
            color=colors[fi], lw=1, label='true')
    ax.plot(pred_pos[:, fi, 0], pred_pos[:, fi, 1],
            '--', color=colors[fi], lw=1, label='pred')
    ax.set(title=f'fish {fi+1}', xlabel='x (px)', ylabel='y (px)', aspect='equal')
    ax.legend(fontsize=8)
fig.suptitle('Spatial Trajectories')
fig.tight_layout()
fig.savefig(f'{PNG_DIR}/fish_spatial.png', dpi=150)
plt.show()